# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2: Refresh / Content Opportunity Scoring** — "Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?"

I'm picking this lane because it turns into a concrete, recurring workflow rather than an open-ended exploration: instead of asking "what's interesting in this data," it asks "what should someone do this week." The repo's own starter pipeline (`scripts/01`–`05`, results in `outputs/model_report.md`) already builds an end-to-end version of exactly this — a baseline rule, a trained model, and a ranked queue with reason codes — which tells me the signals this lane needs (age, freshness, position, engagement) are already sitting in the starter dataset and are learnable, not just theoretical. That gives me a working example to study, compare against, and try to beat or understand more deeply over the next 7 weeks, instead of starting from a blank page.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

lane2_columns = ["content_age_days", "days_since_last_update", "freshness_tier", "avg_position", "engagement_rate", "trend_direction"]

print(f"Rows loaded: {len(df):,}, columns: {df.shape[1]}")
print("Lane 2 signal columns present:", all(c in df.columns for c in lane2_columns))

Rows loaded: 30,000, columns: 44
Lane 2 signal columns present: True


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Given a client's full page inventory, which pages should be prioritized *this* cycle for refresh, expansion, protection, pruning, or monitoring?

**Who acts, and how:** The content strategist working that client account. Instead of choosing pages by gut feel, alphabetical order, or "whatever was published longest ago," they pull the top-N pages off a ranked queue (with reason codes attached) and schedule that work for the cycle.

**Cost of a wrong call:**
- *False positive* (flagged as high-priority but it was actually fine): writer/strategist hours spent refreshing a page that didn't need it — a direct budget cost, and an opportunity cost, since that time didn't go to a page that actually needed the work.
- *False negative* (a real decliner never gets flagged): the page keeps losing traffic and clicks quietly until the next audit cycle catches it, compounding the client's organic traffic loss in the meantime.

**Why data/ML helps at all:** A client can have thousands of pages, and across 32 clients the starter dataset alone has 30,000 rows — far more than anyone can manually review page-by-page every cycle. "Which pages are declining" isn't one clean signal either; it depends on several things at once (content age, freshness, search position, engagement) interacting in ways that are too tangled to hand-code into a single if/else rule. This repo's own baseline-vs-model comparison shows there's real room between a hand-written rule and a learned model: the baseline rule scores ROC-AUC 0.627 / precision@50 0.240, while a random forest on the same data reaches 0.750 / 0.740 (`outputs/model_report.md`). That gap is the case for ML earning its place here — a plain rule leaves real signal on the table.

In [2]:
pages_per_client = df.groupby("client_id").size()

print(f"Average pages per client: {pages_per_client.mean():.0f}")
print(f"Largest client inventory: {pages_per_client.max():,} pages")

Average pages per client: 938
Largest client inventory: 7,008 pages


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Using the `df` loaded in Section 1 (`data/raw/content_refresh_anonymized.csv`), a few more numbers computed live (not copied from any report):

In [3]:
n_rows = len(df)
n_clients = df["client_id"].nunique()
pct_declining = (df["trend_direction"] == "down").mean() * 100
pct_no_position_data = (df["avg_position"] == 0).mean() * 100

print(f"Rows: {n_rows:,} across {n_clients} clients")
print(f"Share trending down (candidate refresh targets): {pct_declining:.1f}%")
print(f"Share with no position data (avg_position == 0): {pct_no_position_data:.1f}%")

Rows: 30,000 across 32 clients
Share trending down (candidate refresh targets): 54.2%
Share with no position data (avg_position == 0): 4.0%


Over half the inventory is trend-down and a further slice has no reliable position data at all — that's a large, real candidate pool that a person can't review one page at a time, and exactly the kind of prioritization problem a ranking approach is meant to sort through.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- An *observed* association between signals available today (content age, freshness, search position, engagement) and past decline, within this 90-day anonymized window.
- A *ranked, decision-support* priority list — meant to focus human review, not to act on its own.
- *Directional* confidence, expressed in tiers (high / medium / low confidence), not certainty.

**What I can't claim:**
- *Causal proof* that refreshing a flagged page will improve its ranking — no experiment (e.g. an A/B test on refreshed vs. untouched pages) has been run, so this stays correlational.
- Anything about *how Google's algorithm works* — the model reflects patterns in this dataset, not Google's ranking logic.
- Results *generalizing beyond these 32 clients or this 90-day window* without further validation on new data.
- `trend_direction` and `trend_pct` will never be used as model features — they're what the decline label is built from, so using them as inputs would be leakage, not signal.

In [4]:
excluded_for_leakage = ["trend_direction", "trend_pct"]
id_columns = ["content_id", "client_id"]
candidate_features = [c for c in df.columns if c not in excluded_for_leakage + id_columns]

print(f"Leakage-excluded columns (define the label, so never features): {excluded_for_leakage}")
print(f"Candidate feature columns remaining: {len(candidate_features)}")

Leakage-excluded columns (define the label, so never features): ['trend_direction', 'trend_pct']
Candidate feature columns remaining: 40


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.